## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xeofs as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
SAVE_FNAME = SAVE_FP / "sst_on_pr_grid.nc"

## load precip
sst_total = xr.open_dataarray(SAVE_FNAME).compute()

# ## fix times (cesm2 labels by end of period)
# sst_total = sst_total.assign_coords(
#     {"time": xr.date_range(start="0850-01", end="2005-12", freq="MS")}
# )

# ## drop beginning for model spinup
# sst_total = sst_total.sel(time=slice("0900-01", None))

## compute mean and annual avg
sst_bar = sst_total.mean(["time", "member"])
sst = sst_total - sst_bar

## remove seasonal cycle (to simplify)
# sst = sst.groupby("time.year").mean()
# sst = sst.resample({"time": "YS"}).mean()

## Compute EOFs

In [ ]:
## truncate to neofs
neofs = 300

## save filepath for eofs
eofs_save_fp = pathlib.Path(SAVE_FP, "snr", "eofs_sst.nc")

## weight data sst by coslat
W = src.utils.get_coslat_weights(sst)
X = sst * W

if eofs_save_fp.is_file():
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    eofs_ = xr.open_dataset(eofs_save_fp, decode_times=time_coder)

else:

    ## specs for EOFs
    eofs_kwargs = dict(n_modes=300, standardize=False, use_coslat=False, center=True)

    ## initialize EOF model
    eofs = xe.single.EOF(**eofs_kwargs)

    ## compute
    eofs.fit(X, dim=["time", "member"])

    ## extract components
    N = int(len(X.member) * len(X.time))
    Q = eofs.scores(normalized=True)
    Z = eofs.components(normalized=True)
    sigma = np.sqrt(N * eofs.explained_variance())

    ## merge data
    eofs_ = xr.merge(
        [
            Z.rename("Z"),
            sigma.rename("sigma"),
            Q.rename("Q"),
            W.rename("W"),
        ]
    )

    ## make sure attributes are empty
    eofs_.attrs = dict()
    for v in list(eofs_):
        eofs_[v].attrs = dict()

    ## save
    eofs_.to_netcdf(eofs_save_fp)


## do truncation
eofs_ = eofs_.isel(mode=slice(None, neofs))

### test things make sense

In [ ]:
## compute inner products
n_testmode = 10
idx = dict(mode=slice(None, n_testmode))
QtQ = xr.dot(
    eofs_["Q"].isel(idx),
    eofs_["Q"].isel(idx).rename({"mode": "mode_out"}),
    dims=["member", "time"],
)

ZtZ = xr.dot(
    eofs_["Z"].isel(idx).fillna(0),
    eofs_["Z"].isel(idx).fillna(0).rename({"mode": "mode_out"}),
    dims=["lat", "lon"],
)

## check they're identities
eye = np.eye(n_testmode)
print(np.allclose(ZtZ.values, eye))
print(np.allclose(QtQ.values, eye))

### check we can reconstruct the data
idx = dict(time=slice(None, 30))

## reconstruct using package and custom
gt = sst.isel(idx)
r1 = (
    1
    / eofs_["W"]
    * xr.dot(eofs_["Z"], eofs_["sigma"], eofs_["Q"].isel(idx), dim="mode")
)

## check reconstruction close to original
print(f"{xr.corr(gt, r1).values.item():.6f}")

## SNR computation

Original data:

\begin{align}
    \mathbf{X} &= \mathbf{Z}\boldsymbol{\Sigma}\mathbf{Q}^T
\end{align}

Eigenvectors:

\begin{align}
    \mathbf{v}_k &= \mathbf{Z}\boldsymbol{\Sigma}^{-1}\mathbf{u}_k
\end{align}

Project ensemble mean data onto eigs:
\begin{align}
    \mathbf{M} &= \boldsymbol{\Sigma}^{-1}\mathbf{Z}^T\left<\mathbf{X}\right> = \left<\boldsymbol{\Sigma}^{-1}\mathbf{Z}^T\mathbf{X}\right> = \left<\mathbf{Q}\right>
\end{align}

Project all data onto $k^{th}$ learned component
\begin{align}
    \mathbf{v}_k^T\mathbf{X} &= \mathbf{u}_k^T\boldsymbol{\Sigma}^{-1}\mathbf{Z}^T\mathbf{X}\\
    &= \mathbf{u}_k^T\mathbf{Q}^T
\end{align}

Spatial patterns:
Choose $\mathbf{P}$ to satisfy:
\begin{align}
    \mathbf{X} &\approx \mathbf{PV}^T\mathbf{X} = \mathbf{PU}^T\mathbf{Q}^T\\
    \implies \mathbf{P} &= \mathbf{X Q U} = \mathbf{Z}\boldsymbol{\Sigma} \mathbf{U}\\
    \mathbf{p}_k &= \mathbf{Z}\boldsymbol{\Sigma}\mathbf{u}_k
\end{align}

In [ ]:
def dot(x, y, **kwargs):
    """xr.dot wrapper which skips na values"""
    return xr.dot(x.fillna(0), y.fillna(0), **kwargs)

Compute covariance matrix

In [ ]:
DO_TESTS = False

## project ensemble-mean data on EOFs
eofs_["Qbar"] = eofs_["Q"].mean("member")

## covariance
ntime = len(eofs_.time)
QQt = (
    1
    / (ntime - 1)
    * (eofs_["Qbar"] * eofs_["Qbar"].rename({"mode": "mode_"})).sum("time")
)

## eigendecomp
Lambda_sqr, U = np.linalg.eigh(QQt.values)
U = U[:, ::-1]
Lambda_sqr = Lambda_sqr[::-1]

## put in xarray dataset
eofs_snr = xr.Dataset(
    data_vars=dict(
        U=(("mode", "mode_snr"), U),
        Lambda=("mode_snr", np.sqrt(Lambda_sqr)),
    ),
    coords=dict(mode=eofs_.mode.values, mode_snr=eofs_.mode.values),
)

## get timeseries timeseries
eofs_snr["tk"] = dot(eofs_snr["U"], eofs_["Q"], dim="mode")
eofs_snr["tk_bar"] = dot(eofs_snr["U"], eofs_["Qbar"], dim="mode")

# get patterns
eofs_snr["P"] = xr.dot(
    eofs_["Z"].fillna(0),
    eofs_["sigma"],
    eofs_snr["U"],
    dim="mode",
)

## test a different way of computing Qbar, tk, P
if DO_TESTS:
    eofs_["Qbar_test"] = (
        1 / eofs_["sigma"] * dot(eofs_["Z"], X.mean("member"), dim=["lat", "lon"])
    )

    eofs_snr["P_test"] = dot(X, eofs_snr["tk"], dim=["time", "member"])

    ## check tk's are orthgonal
    TtT = xr.dot(
        eofs_snr["tk"],
        eofs_snr["tk"].rename({"mode_snr": "mode"}),
        dim=["time", "member"],
    )

    ## do tests
    print(np.allclose(eofs_["Qbar"], eofs_["Qbar_test"], atol=1e-2))
    print(np.allclose(eofs_snr["P"], eofs_snr["P_test"]))
    print(np.allclose(TtT.values, np.eye(TtT.shape[0])))

In [ ]:
## filtered ensemble mean
## choose cutoff based on eigenvalue spectrum, or check significant patterns with bootstrapping
# 5 works for annual mean
# 15 for seasonally-varying?
M = 15  # number of forced patterns to retain,

## func to truncate
trunc = lambda x: x.isel(mode_snr=slice(None, M))

## do truncation
eofs_snr["X_trunc"] = (trunc(eofs_snr["tk_bar"]) * trunc(eofs_snr["P"])).sum("mode_snr")

## apply latitude weighting (reconstruct original data)
forced = eofs_snr["X_trunc"] / eofs_["W"]
anom = (X - eofs_snr["X_trunc"]) / eofs_["W"]

## merge data
data_split = xr.merge([forced.rename("forced"), anom.rename("anom")])

## save to file
snr_fname = pathlib.Path(SAVE_FP, "snr", "sst_split.nc")
if snr_fname.is_file():
    pass
else:
    data_split.to_netcdf(snr_fname)

## Plot results

### Spatial patterns

In [ ]:
## specify intervals
dxs = np.array([2])

fig = plt.figure(figsize=(7, 7), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=25, lon_range=(60, 280))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)


for j in range(3):
    cp00 = plot_pr_diff(
        axs[j, 0],
        eofs_snr["P"].isel(mode_snr=j).rename({"lat": "latitude", "lon": "longitude"}),
        dx=5e1,
    )

    ## label
    axs[j, 0].set_title(f"Mode {j}", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[65, 150], lats=[-25, 25], c="k", ls="--")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
# fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
add_gridlines(axs)
plt.show()

### Signal to noise

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
# ax.plot(np.cumsum(eofs_snr["Lambda"])/(eofs_snr["Lambda"]).sum(), ls="--")
ax.plot(eofs_snr["Lambda"], ls="--")
ax.set_xlim([-0.5, 50])
# src.utils.add_axes([ax])
ax.set_xticks([0, 10])
src.utils.add_vticks([ax], xticks=[0, M - 1, 30], xlines=[0, M - 1])
# src.utils.add_hticks([ax], yticks=[0, 0.007], ylines=[0])
ax.set_ylim([-0.001, None])
# ax.set_yticks([0, 10])
plt.show()

### Timeseries

#### specify some funcs to plot

In [ ]:
resamp = lambda x: x.groupby("time.year").mean()

In [ ]:
## compute ensemble mean stuff for comparison
sst_bar = sst.mean("member")

## funcs
global_avg = lambda x: resamp(x.mean(["lat", "lon"]))
eq_avg = lambda x: resamp(x.sel(lat=slice(-5, 5)).mean("lat"))
get_Tw = lambda x: eq_avg(x.sel(lon=slice(120, 170)).mean("lon"))
get_Te = lambda x: eq_avg(x.sel(lon=slice(230, 280)).mean("lon"))
get_T34 = lambda x: eq_avg(x.sel(lon=slice(190, 240)).mean("lon"))
get_dTdx = lambda x: get_Te(x) - get_Tw(x)


# sst_avg = sst.mean(["member", "lat", "lon"])

# fig, ax = plt.subplots(figsize=(8, 3))
# ax.plot(sst_avg)
# src.utils.add_hticks([ax], yticks=[-1, 0, 0.5], ylines=[0])
# plt.show()

## Timeseries

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")

# axs[0].plot(eofs_snr.time.dt.year, -eofs_snr["tk_bar"].isel(mode_snr=0))
# axs[1].plot(eofs_snr.time.dt.year, global_avg(sst_bar), c=sns.color_palette()[1])
axs[0].plot(-resamp(eofs_snr["tk_bar"]).isel(mode_snr=4))
axs[1].plot(global_avg(sst_bar), c=sns.color_palette()[1])

src.utils.add_hticks(axs, ylines=[0], yticks=[0])
axs[0].set_title("SNR mode")
axs[1].set_title("SST glob. avg")
# axs[

# ax.plot(T, -eofs_snr["tk_bar"].isel(mode_snr=0).values, ls="--")
plt.show()

Filtered global SST

In [ ]:
yr = resamp(sst_bar).year

kwargs = dict(lw=2)

fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.set_title(r"global avg")
ax.plot(yr, global_avg(sst_bar), **kwargs)
ax.plot(yr, global_avg(eofs_snr["X_trunc"]), **kwargs)
ax.set_xticks([900, 1200, 2005])
src.utils.add_hticks([ax], yticks=[-0.5, 0, 0.25], ylines=[-0.5, 0, 0.25])
plt.show()

fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.set_title(r"$dT/dx$")
ax.plot(yr, get_dTdx(sst_bar), **kwargs)
ax.plot(yr, get_dTdx(eofs_snr["X_trunc"]), **kwargs)
ax.set_xticks([900, 1200, 2005])
src.utils.add_hticks([ax], yticks=[-0.5, 0, 0.25], ylines=[-0.5, 0, 0.25])
plt.show()

fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.set_title(r"$T_w$")
ax.plot(yr, get_Tw(sst_bar), **kwargs)
ax.plot(yr, get_Tw(eofs_snr["X_trunc"]), **kwargs)
ax.set_xticks([900, 1200, 2005])
src.utils.add_hticks([ax], yticks=[-0.5, 0, 0.25], ylines=[-0.5, 0, 0.25])
plt.show()

fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.set_title(r"$T_e$")
ax.plot(yr, get_Te(sst_bar), **kwargs)
ax.plot(yr, get_Te(eofs_snr["X_trunc"]), **kwargs)
ax.set_xticks([900, 1200, 2005])
src.utils.add_hticks([ax], yticks=[-0.5, 0, 0.25], ylines=[-0.5, 0, 0.25])
plt.show()

fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.set_title(r"$T_{34}$")
ax.plot(yr, get_T34(sst_bar), **kwargs)
ax.plot(yr, get_T34(eofs_snr["X_trunc"]), **kwargs)
ax.set_xticks([900, 1200, 2005])
src.utils.add_hticks([ax], yticks=[-0.5, 0, 0.25], ylines=[-0.5, 0, 0.25])
plt.show()

# plt.plot(X_trunc.mean(["member", "lat", "lon"]))

In [ ]:
eofs_["W"].max()